# Random Forest training on Instagram_cleaned.csv

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve, auc
import joblib
import json

In [15]:
DATA = Path('..') / 'dataset' / 'cleaned' / 'Instagram_cleaned.csv'
ARTIFACTS = Path('..') / 'artifacts'
REPORTS = Path('..') / 'reports'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

In [16]:
df = pd.read_csv(DATA)
df.shape

(5000, 68)

In [17]:
assert 'fake' in df.columns, 'Target column `fake` not found'
y = df['fake'].astype(int)
feature_cols = [
    c for c in df.columns
    if (
        c.endswith('_capped')
        or c.startswith('log_')
        or c.endswith('_clipped')
    )
    and c not in ['fake', 'fake_capped']
]
for flag in ['profile_pic', 'private', 'external_url', 'name_eq_username']:
    if flag in df.columns and flag not in feature_cols:
        feature_cols.append(flag)
print('Using', len(feature_cols), 'features')
print(feature_cols)

X = df[feature_cols].fillna(0)

Using 19 features
['profile_pic_capped', 'username_num_length_capped', 'fullname_words_capped', 'fullname_num_length_capped', 'name_eq_username_capped', 'description_length_capped', 'external_url_capped', 'private_capped', 'num_posts_capped', 'num_followers_capped', 'num_follows_capped', 'followers_following_ratio_capped', 'log_num_followers', 'log_num_posts', 'followers_following_ratio_clipped', 'profile_pic', 'private', 'external_url', 'name_eq_username']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train shape', X_train.shape, 'Test shape', X_test.shape)
print('Train class distribution:', y_train.value_counts(normalize=True))

Train shape (4000, 19) Test shape (1000, 19)
Train class distribution: fake
1    0.5
0    0.5
Name: proportion, dtype: float64


In [19]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:,1]
metrics = dict(accuracy=accuracy_score(y_test,y_pred), precision=precision_score(y_test,y_pred,zero_division=0), recall=recall_score(y_test,y_pred,zero_division=0), f1=f1_score(y_test,y_pred,zero_division=0), roc_auc=roc_auc_score(y_test,y_proba))
metrics

{'accuracy': 0.989,
 'precision': 0.9919517102615694,
 'recall': 0.986,
 'f1': 0.9889669007021064,
 'roc_auc': 0.999646}

In [22]:
model_path = ARTIFACTS / 'rf_model.pkl'
joblib.dump(rf, model_path)
fi = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
fi.to_csv(ARTIFACTS / 'rf_feature_importances.csv', index=False)
with open(REPORTS / 'rf_evaluation.md', 'w') as f:
    f.write('# Random Forest evaluation')
    f.write(json.dumps(metrics, indent=2))
print('Saved model to', model_path)
fi.head()

Saved model to ..\artifacts\rf_model.pkl


,feature,importance
0,profile_pic_capped,0.162540
15,profile_pic,0.157534
12,log_num_followers,0.101984
13,log_num_posts,0.096058
1,username_num_length_capped,0.092549


In [27]:
sample_probs = rf.predict_proba(X_test)[:, 1]
print(sample_probs[:10])

[0.    1.    0.04  0.995 1.    1.    1.    0.995 0.    0.015]


## Fraud Risk Scores

In [28]:
# Compute fraud risk for the full dataset and save augmented CSV
# X contains the selected features for modelling (from earlier cells)
X_full = df[feature_cols].fillna(0)
# predict probabilities (positive class)
probs = rf.predict_proba(X_full)[:, 1]
df['fraud_risk_prob'] = probs
df['fraud_risk_percent'] = (df['fraud_risk_prob'] * 100).round(2)
# outlier count: sum of *_outlier_p columns per row
outlier_cols = [c for c in df.columns if c.endswith('_outlier_p')]
if outlier_cols:
    df['outlier_count'] = df[outlier_cols].sum(axis=1)
else:
    df['outlier_count'] = 0
# suspicion score: fraud percent + 5 * outlier_count, capped at 100
df['suspicion_score'] = (df['fraud_risk_percent'] + 5 * df['outlier_count']).clip(upper=100).round(2)
# save augmented dataframe
out_path = Path('..') / 'dataset' / 'cleaned' / 'Instagram_with_risk_scores.csv'
df.to_csv(out_path, index=False)
print('Saved augmented CSV to', out_path)
df[['fraud_risk_prob','fraud_risk_percent','outlier_count','suspicion_score']].head()

Saved augmented CSV to ..\dataset\cleaned\Instagram_with_risk_scores.csv


,fraud_risk_prob,fraud_risk_percent,outlier_count,suspicion_score
0,0.030,3.0,0,3.0
1,0.000,0.0,0,0.0
2,0.000,0.0,0,0.0
3,0.005,0.5,0,0.5
4,0.000,0.0,0,0.0


In [29]:
# Rule-based explanation function
def generate_reasons(row):
    reasons = []
    if row.get('profile_pic') == 0:
        reasons.append('No profile picture')
    if row.get('username_num_length_capped', 0) > 0.5:
        reasons.append('Digit-heavy username')
    if row.get('description_length_capped') == 0:
        reasons.append('Empty bio')
    if row.get('followers_following_ratio_clipped', 1.0) < 0.2:
        reasons.append('Suspicious follower/following ratio')
    if row.get('num_posts_capped', 9999) < 5:
        reasons.append('Very low posting activity')
    return reasons

# Apply rules and save reasons column
df['reasons'] = df.apply(generate_reasons, axis=1)
df['reasons'] = df['reasons'].apply(lambda x: '; '.join(x) if isinstance(x, (list, tuple)) and x else '')
out_path = Path('..') / 'dataset' / 'cleaned' / 'Instagram_with_risk_scores.csv'
df.to_csv(out_path, index=False)
print('Saved augmented CSV with reasons to', out_path)
df[['fraud_risk_percent','suspicion_score','reasons']].head()

Saved augmented CSV with reasons to ..\dataset\cleaned\Instagram_with_risk_scores.csv


,fraud_risk_percent,suspicion_score,reasons
0,3.0,3.0,
1,0.0,0.0,
2,0.0,0.0,Empty bio
3,0.5,0.5,
4,0.0,0.0,Empty bio


In [33]:
#Display sample profiles with risk scores and reasons
for i in range(10):
    sample_row = X_test.iloc[i]
    risk = round(sample_probs[i] * 100, 2)
    reasons = generate_reasons(sample_row)

    print(f"\nProfile {i}")
    print("Risk Score:", risk, "%")
    print("Reasons:", reasons)


Profile 0
Risk Score: 0.0 %
Reasons: []

Profile 1
Risk Score: 100.0 %
Reasons: ['No profile picture', 'Suspicious follower/following ratio', 'Very low posting activity']

Profile 2
Risk Score: 4.0 %
Reasons: []

Profile 3
Risk Score: 99.5 %
Reasons: ['Digit-heavy username']

Profile 4
Risk Score: 100.0 %
Reasons: ['No profile picture', 'Empty bio', 'Very low posting activity']

Profile 5
Risk Score: 100.0 %
Reasons: ['No profile picture', 'Empty bio', 'Suspicious follower/following ratio', 'Very low posting activity']

Profile 6
Risk Score: 100.0 %
Reasons: ['No profile picture', 'Empty bio', 'Very low posting activity']

Profile 7
Risk Score: 99.5 %
Reasons: ['No profile picture', 'Empty bio', 'Very low posting activity']

Profile 8
Risk Score: 0.0 %
Reasons: []

Profile 9
Risk Score: 1.5 %
Reasons: ['Empty bio']
